The following script shows how to extract OpenET Reference ET bias correction factors for each unique gridMET cell provided in a .csv. 

The output .txt file is used in the ET Demands workflow to apply adjustment factors to the reference ET estimates based on the gridMET weather data. Adjustments are applied to account for evaportive cooling effects not represented in gridMET. 

In [ ]:
#import python packages
from functools import reduce
import pandas as pd
import os
import ee

In [ ]:
gcloud_project_id = 'ENTER GCP HERE'

# intialize gee
ee.Initialize(project=gcloud_project_id)

In [ ]:
#Input .csv containing list of gridMET cells
 = r'nv_et_cells_pts.csv'

#read .csv into pandas dataframe
input_df = pd.read_csv(gridmet_csv_path)

# eto or etr
ref_type = 'etr'

# Opent ET location of the v1 bias correction image collection
bias_ic_path = "projects/openet/reference_et/gridmet/ratios/v1/monthly/{}".format(ref_type)

In [ ]:
print('Extracting {} ratios.'.format(ref_type))

bias_ic = ee.ImageCollection(bias_ic_path)

def data_frame(feature_collection):
    features = feature_collection.getInfo()['features']
    output_data = []
    for f in features:
        # Store all attributes in a dict
        attr = f['properties']
        output_data.append(attr)
    output_df = pd.DataFrame(output_data)
    return output_df

final_df = None
out_df = []

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct', 'Nov','Dec']

#Loop through dataframe row by row
for input_i, input_row in input_df.iterrows():
    lat = input_row.LAT
    lon = input_row.LON
    id = input_row.GRIDMET_ID
    print('Processing: {}'.format(id))

    def reduce_image(image):
        values = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=ee.Geometry.Point(lon, lat).buffer(100),
            scale=100)

        return ee.Feature(None,
                          {"Met Node ID": id,
                           "month": image.get("month_abbrev"),
                           "ETo": values.get("b1")
                           })
    data = data_frame(bias_ic.map(reduce_image))

    if final_df is None:
        final_df = data
    else:
        final_df = pd.concat([final_df, data])

for month in months:
    data_month = (final_df.loc[data.month == month, ["ETo", "Met Node ID"]])
    data_month = data_month.rename(columns={"ETo": month})
    out_df.append(data_month)

df_merged = reduce(lambda  left,right: pd.merge(left, right, on=['Met Node ID'],
                                            how='outer'), out_df)

df_merged["Met Node Name"] = ""

df_merged = df_merged[["Met Node ID", "Met Node Name",
                       'Jan','Feb','Mar','Apr','May','Jun','Jul',
                       'Aug','Sep','Oct','Nov','Dec']]
  

In [ ]:
#Save the dataframe to a .txt after dataframe download is complete. 
file_out = 'OpenET_v1_{}_ratios_OR_v1.txt'.format(ref_type)         
print('Saving file: {}'.format(file_out))
df_merged.to_csv(file_out, index=False, header=True, sep="\t", mode="a")